## Changelog
- parent: 20260507_214919_cd6c065b
- change: revert garage-PLS (proved counter-productive on the run-14 baseline,
    +154.56 LB) and tune GradientBoostingRegressor over a small 4-axis grid:
    learning_rate (0.05, 0.1), n_estimators (300, 600), max_depth (3, 4, 5),
    subsample (0.8, 1.0). 24 configs × 5-fold CV.
- hypothesis: at this baseline the residual margin from preprocessing/FE is
    small (recap 2026-05-07). The remaining lever is model capacity. Lower
    learning_rate + more estimators averages noisier per-tree contributions;
    deeper trees (4-5) capture interaction terms the 3-deep default cannot;
    subsample=0.8 introduces stochastic regularization. Each axis individually
    moves CV by little, but the joint search is the cheap way to find a
    cooperating combination.

In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")

In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

# Drop Id (row identifier, no signal) and SalePrice (target). Everything else
# goes through preprocessing.
DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

# Auto-detect num/cat from train; apply the same split to test.
NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold

from utils.ames_sklearn_pipeline import AmesNAImputer, AmesEncoder, TargetEncodeColumn
from utils.ames_feature_engineering import add_size_features, add_temporal_features

pipe = Pipeline([
    ("na",      AmesNAImputer()),
    ("size",    FunctionTransformer(
                    add_size_features,
                    kw_args={"drop_originals": True},
                )),
    ("fe",      FunctionTransformer(
                    add_temporal_features,
                    kw_args={"drop_originals": True},
                )),
    ("nbhd_te", TargetEncodeColumn("Neighborhood")),
    ("encoder", AmesEncoder()),
    ("model",   GradientBoostingRegressor(random_state=42)),
])

param_grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__n_estimators":  [300, 600],
    "model__max_depth":     [3, 4, 5],
    "model__subsample":     [0.8, 1.0],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipe, param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)
search.fit(X, y)

print(f"Best CV RMSE (log-price): {-search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

# top 5 configs by mean CV score for landscape inspection
results = pd.DataFrame(search.cv_results_)
top = (
    results[[
        "param_model__learning_rate", "param_model__n_estimators",
        "param_model__max_depth",     "param_model__subsample",
        "mean_test_score",            "std_test_score",
    ]]
    .assign(mean_rmse=lambda d: -d["mean_test_score"])
    .sort_values("mean_rmse")
    .head(5)
    .drop(columns=["mean_test_score"])
)
print("\nTop 5 configs:")
print(top.to_string(index=False))

# refit best estimator is already done by GridSearchCV(refit=True)
pipe = search.best_estimator_

In [ ]:
test_pred = np.expm1(pipe.predict(X_test))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)